# RAG Local: Chat con Documentos 100% Local

## ¿Qué es RAG?

**RAG (Retrieval Augmented Generation)** es una técnica que combina búsqueda de información con generación de texto. Permite que un modelo de lenguaje responda preguntas basándose en documentos específicos, no solo en su conocimiento pre-entrenado.

### Flujo RAG:

1. **Ingesta**: Cargar y dividir documentos en fragmentos (chunks)
2. **Embeddings**: Convertir texto a vectores numéricos
3. **Vector Store**: Almacenar vectores en una base de datos vectorial
4. **Retrieval**: Buscar los fragmentos más relevantes para una pregunta
5. **Generation**: Usar el contexto recuperado + la pregunta para generar una respuesta

### Ventajas de RAG Local:

- ✅ **Privacidad total**: Todo se procesa localmente
- ✅ **Sin costos de API**: No necesitas servicios externos
- ✅ **Control completo**: Puedes usar tus propios documentos
- ✅ **Rápido**: Sin latencia de red

En este notebook construirás un sistema RAG completo usando modelos locales.


In [ ]:
# Setup inicial
import sys
import os
from pathlib import Path

# Hack para importar desde src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.models import get_local_llm, get_local_embeddings

print("✓ Imports completados")


## Setup y Datos de Ejemplo

Primero, crearemos un archivo de texto de ejemplo con contenido técnico. Esto nos permitirá probar el sistema RAG sin necesidad de subir archivos externos.


In [ ]:
# Crear directorio data si no existe
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

# Crear archivo de ejemplo con contenido técnico
sample_file = data_dir / "sample.txt"
sample_content = """POLÍTICAS DE SEGURIDAD DE LA EMPRESA X

1. ACCESO A SISTEMAS
Todos los empleados deben usar autenticación de dos factores (2FA) para acceder a los sistemas corporativos.
Las contraseñas deben tener al menos 12 caracteres e incluir mayúsculas, minúsculas, números y símbolos.
El acceso remoto solo está permitido a través de VPN corporativa.

2. PROTECCIÓN DE DATOS
Los datos sensibles deben estar encriptados tanto en tránsito como en reposo.
No se permite almacenar información confidencial en servicios de almacenamiento en la nube no autorizados.
Todos los dispositivos móviles deben tener cifrado de disco completo activado.

3. GESTIÓN DE INCIDENTES
Cualquier sospecha de brecha de seguridad debe reportarse inmediatamente al equipo de seguridad.
El tiempo de respuesta objetivo para incidentes críticos es de 15 minutos.
Todos los incidentes deben documentarse en el sistema de tickets.

4. CAPACITACIÓN
Todos los empleados deben completar el curso de seguridad anual.
Se realizarán simulacros de phishing trimestrales.
El cumplimiento de las políticas de seguridad es obligatorio para todos los empleados.

5. DISPOSITIVOS
Los dispositivos personales (BYOD) requieren aprobación previa del departamento de TI.
Todos los dispositivos deben tener software antivirus actualizado.
No se permite instalar software no autorizado en dispositivos corporativos.

6. AUDITORÍA
Se realizan auditorías de seguridad trimestrales.
Los logs de acceso se conservan durante 12 meses.
Cualquier actividad sospechosa se investiga de inmediato.
"""

# Escribir el archivo
with open(sample_file, "w", encoding="utf-8") as f:
    f.write(sample_content)

print(f"✓ Archivo de ejemplo creado: {sample_file}")
print(f"  Tamaño: {len(sample_content)} caracteres")


## Paso 1: Ingesta y División de Texto

### ¿Por qué dividir el texto?

Los modelos de lenguaje tienen límites en el tamaño del contexto (context window). Además, para búsqueda semántica, es más efectivo trabajar con fragmentos pequeños y relevantes que con documentos completos.

Usaremos `RecursiveCharacterTextSplitter` que divide el texto de forma inteligente, intentando mantener párrafos y oraciones completas.


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Cargar el documento
loader = TextLoader(str(sample_file), encoding="utf-8")
documents = loader.load()

print(f"✓ Documento cargado: {len(documents)} documento(s)")
print(f"  Tamaño original: {len(documents[0].page_content)} caracteres")

# Dividir en chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Tamaño máximo de cada chunk
    chunk_overlap=50,   # Solapamiento entre chunks (mantiene contexto)
    length_function=len,
)

chunks = text_splitter.split_documents(documents)

print(f"\n✓ Documento dividido en {len(chunks)} chunks")
print(f"\nEjemplo de chunk (primeros 200 caracteres):")
print("-" * 60)
print(chunks[0].page_content[:200] + "...")


## Paso 2: Embeddings y Vector Store

### ¿Qué son los Embeddings?

Los **embeddings** son representaciones numéricas (vectores) del texto que capturan su significado semántico. Textos similares tienen vectores similares, lo que permite búsqueda semántica.

### Vector Store

Una **base de datos vectorial** almacena estos embeddings y permite buscar documentos similares usando distancia coseno o producto punto.

Usaremos **ChromaDB** como base de datos vectorial local y efímera.


In [ ]:
from langchain_community.vectorstores import Chroma
import shutil

# Limpiar cualquier instancia previa de ChromaDB (importante en Windows)
chroma_path = "../data/chroma_db"
if os.path.exists(chroma_path):
    try:
        shutil.rmtree(chroma_path)
        print("✓ Instancia previa de ChromaDB limpiada")
    except Exception as e:
        print(f"⚠ Advertencia al limpiar: {e}")

# Instanciar embeddings locales
embeddings = get_local_embeddings()

print(f"✓ Embeddings configurados: {embeddings.model_name}")

# Crear la base de datos vectorial
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(chroma_path)  # Opcional: persistir en disco
)

print(f"✓ Vector Store creado con {len(chunks)} documentos")
print(f"  Dimensión de embeddings: {embeddings.client.get_sentence_embedding_dimension()}")


## Paso 3: El Retriever

El **Retriever** es un componente que busca los documentos más relevantes para una consulta. Convierte la pregunta en un embedding, busca los chunks más similares en la base de datos vectorial y los devuelve.


In [ ]:
# Convertir el vector store en un retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}  # Devolver los 3 chunks más relevantes
)

print("✓ Retriever configurado (top 3 resultados)")

# Probar el retriever
pregunta_test = "¿Cuáles son las políticas de contraseñas?"
docs_recuperados = retriever.invoke(pregunta_test)

print(f"\n✓ Búsqueda realizada para: '{pregunta_test}'")
print(f"  Documentos recuperados: {len(docs_recuperados)}")
print("\n" + "="*60)
print("CONTENIDO RECUPERADO:")
print("="*60)
for i, doc in enumerate(docs_recuperados, 1):
    print(f"\n--- Documento {i} ---")
    print(doc.page_content[:300] + "..." if len(doc.page_content) > 300 else doc.page_content)


## Paso 4: La Cadena RAG (The LCEL Magic)

Ahora construiremos la cadena RAG completa usando LCEL. La magia está en cómo combinamos:

1. **Retriever**: Busca el contexto relevante
2. **Prompt**: Formatea la pregunta con el contexto
3. **Model**: Genera la respuesta
4. **Parser**: Extrae el texto limpio

### El Prompt RAG

El prompt típico de RAG incluye:
- **Contexto**: Los documentos recuperados
- **Pregunta**: La pregunta del usuario
- **Instrucciones**: Cómo usar el contexto para responder


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Crear el prompt RAG
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente experto que responde preguntas basándote ÚNICAMENTE en el contexto proporcionado.
    
Si la respuesta no está en el contexto, di que no tienes esa información.
Responde de forma clara y concisa."""),
    ("human", """Contexto:
{context}

Pregunta: {question}

Respuesta:""")
])

print("✓ Prompt RAG creado")


### Construir la Cadena con LCEL

La clave está en usar `RunnablePassthrough` para combinar el retriever con la pregunta:

```python
{
    "context": retriever,                    # Busca documentos relevantes
    "question": RunnablePassthrough()        # Pasa la pregunta tal cual
}
```

Esto crea un diccionario con:
- `context`: Los documentos recuperados (convertidos a texto)
- `question`: La pregunta original

Luego el prompt formatea ambos y el modelo genera la respuesta.


In [ ]:
# Instanciar el modelo y parser
llm = get_local_llm()
output_parser = StrOutputParser()

# Función para formatear los documentos recuperados
def format_docs(docs):
    """Convierte una lista de documentos en un string formateado."""
    return "\n\n".join(doc.page_content for doc in docs)

# Construir la cadena RAG completa usando LCEL
rag_chain = (
    {
        "context": retriever | format_docs,  # Retriever -> formatear docs
        "question": RunnablePassthrough()      # Pasar pregunta tal cual
    }
    | prompt_template                          # Formatear prompt
    | llm                                      # Generar respuesta
    | output_parser                            # Extraer texto
)

print("✓ Cadena RAG construida")
print("\nEstructura:")
print("  Input (pregunta)")
print("    ↓")
print("  Retriever → format_docs → context")
print("  question (passthrough)")
print("    ↓")
print("  Prompt Template")
print("    ↓")
print("  LLM")
print("    ↓")
print("  Output Parser")
print("    ↓")
print("  Output (respuesta)")


### Probar la Cadena RAG

Ahora podemos hacer preguntas y obtener respuestas basadas en el documento.


In [ ]:
# Hacer una pregunta
pregunta = "¿Cuáles son los requisitos de contraseñas?"

print("Pregunta:", pregunta)
print("\n" + "="*60)
print("RESPUESTA:")
print("="*60)

respuesta = rag_chain.invoke(pregunta)
print(respuesta)


In [ ]:
# Otra pregunta
pregunta2 = "¿Qué hacer en caso de una brecha de seguridad?"

print("Pregunta:", pregunta2)
print("\n" + "="*60)
print("RESPUESTA:")
print("="*60)

respuesta2 = rag_chain.invoke(pregunta2)
print(respuesta2)


### Entender el Flujo Completo

Veamos qué está pasando internamente:


In [ ]:
# Desglosar el proceso paso a paso
pregunta_demo = "¿Cuánto tiempo se conservan los logs?"

print("="*60)
print("PROCESO RAG PASO A PASO")
print("="*60)

# Paso 1: Retrieval
print("\n1. RETRIEVAL:")
docs = retriever.invoke(pregunta_demo)
print(f"   → Encontrados {len(docs)} documentos relevantes")

# Paso 2: Formatear contexto
contexto = format_docs(docs)
print(f"   → Contexto formateado ({len(contexto)} caracteres)")

# Paso 3: Formatear prompt
mensajes = prompt_template.invoke({
    "context": contexto,
    "question": pregunta_demo
})
print(f"   → Prompt formateado con {len(mensajes.messages)} mensajes")

# Paso 4: Generar respuesta
respuesta_llm = llm.invoke(mensajes.messages)
print(f"   → Respuesta generada: {len(respuesta_llm.content)} caracteres")

# Paso 5: Parsear
respuesta_final = output_parser.invoke(respuesta_llm)
print(f"   → Texto extraído: {len(respuesta_final)} caracteres")

print("\n" + "="*60)
print("RESPUESTA FINAL:")
print("="*60)
print(respuesta_final)


## Resumen: Componentes RAG

1. **Document Loader**: Carga documentos desde archivos
2. **Text Splitter**: Divide documentos en chunks manejables
3. **Embeddings**: Convierte texto a vectores numéricos
4. **Vector Store**: Almacena y busca embeddings
5. **Retriever**: Busca documentos relevantes para una pregunta
6. **Prompt Template**: Formatea pregunta + contexto
7. **LLM**: Genera respuesta basada en el contexto
8. **Output Parser**: Extrae texto limpio

### Ventajas de LCEL en RAG:

- ✅ **Composición flexible**: Puedes agregar/quitar componentes fácilmente
- ✅ **Debugging fácil**: Puedes invocar cada componente individualmente
- ✅ **Reutilización**: Los componentes se pueden usar en múltiples cadenas
- ✅ **Streaming**: Soporte nativo para respuestas en tiempo real


## 🎯 Ejercicio Práctico

### Tu Turno

1. **Modifica el archivo de datos**:
   - Abre `data/sample.txt`
   - Reemplaza el contenido con información de tu interés (puede ser sobre cualquier tema: recetas, tecnología, historia, etc.)
   - Guarda el archivo

2. **Reconstruye el sistema RAG**:
   - Ejecuta nuevamente las celdas de carga y división de documentos
   - Reconstruye el vector store con los nuevos datos
   - Haz preguntas sobre tu nuevo contenido

3. **Experimenta**:
   - Prueba diferentes tamaños de chunks (`chunk_size`)
   - Cambia el número de documentos recuperados (`k` en el retriever)
   - Modifica el prompt para obtener diferentes estilos de respuesta

### Preguntas para probar:

- Haz preguntas específicas sobre el contenido
- Prueba preguntas que NO estén en el documento (debería decir que no tiene esa información)
- Compara respuestas con diferentes valores de `k` (1, 3, 5)

¡Experimenta y aprende! 🚀


In [ ]:
# Tu código aquí
# 1. Modifica data/sample.txt con tu propio contenido
# 2. Re-ejecuta las celdas de carga y vector store
# 3. Prueba hacer preguntas sobre tu nuevo contenido

# Ejemplo de pregunta:
# pregunta_ejercicio = "Tu pregunta aquí"
# respuesta_ejercicio = rag_chain.invoke(pregunta_ejercicio)
# print(respuesta_ejercicio)



## Notas Adicionales

### Limpieza de ChromaDB

Si necesitas limpiar la base de datos vectorial:

```python
import shutil
chroma_path = "../data/chroma_db"
if os.path.exists(chroma_path):
    shutil.rmtree(chroma_path)
```

### Optimizaciones Futuras

- **Persistencia**: ChromaDB puede guardarse en disco para reutilizar entre sesiones
- **Múltiples documentos**: Puedes cargar múltiples archivos y combinarlos
- **Metadata filtering**: Filtrar búsquedas por metadatos (fuente, fecha, etc.)
- **Hybrid search**: Combinar búsqueda semántica con búsqueda por palabras clave
- **Streaming**: Usar `.stream()` en lugar de `.invoke()` para respuestas en tiempo real

### Troubleshooting

- **Error de bloqueo en Windows**: Asegúrate de limpiar `chroma_db` antes de recrearlo
- **Memoria**: Documentos muy grandes pueden requerir más RAM
- **Embeddings lentos**: La primera vez puede ser lenta (descarga del modelo)
